# Erase — capabilities walkthrough

This notebook runs Bria **Erase** (object removal / inpainting) end to end: install the **`erase`** package from **AWS CodeArtifact**, let it pull the model weights from Hugging Face, then remove a masked object from a sample image with the in-process two-stage pipeline (LaMa coarse fill → SDXL ControlNet refine).

**Environment:** an NVIDIA GPU with ≥ 16 GB VRAM (production runs on **A10G**; any Ampere-or-newer card works), Python **3.10+**. Unlike Increase Resolution this is **not** a TensorRT engine — it runs the standard `torch` + `diffusers` stack, so the runtime is portable across recent CUDA versions.

**Prerequisites:** `BRIA_API_TOKEN` (for the CodeArtifact credential) and network access to the Bria Engine, AWS CodeArtifact, and Hugging Face.

## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`bria-erase`** repository.

In [ ]:
import os
import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN before running this notebook.")

resp = requests.get(
    "https://engine.prod.bria-api.com/v2/auth/access/code_artifact",
    params={"repository": "bria-erase"},
    headers={"api_token": BRIA_API_TOKEN},
    timeout=60,
)
resp.raise_for_status()
result = resp.json()["result"]
CODE_ARTIFACT_PASSWORD = result["authorization_token"].strip()
print("CodeArtifact credential acquired. Expires:", result.get("expiration"))

## 2. Install `erase`

Uses the CodeArtifact token as the password for the `bria-erase` PyPI index (username `aws`). `torch`, `diffusers`, `opencv`, etc. resolve automatically from PyPI — there is no CPU/GPU role split.

In [ ]:
import subprocess, sys
from urllib.parse import quote

CA = (
    "https://aws:" + quote(CODE_ARTIFACT_PASSWORD, safe="")
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-erase/simple/"
)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "erase",
    "--extra-index-url", CA,
])

## 3. Load the sample image + mask

The mask is white (255) over the region to erase. If it does not match the image size it is resized with nearest-neighbour, then cleaned to a hard binary mask.

In [ ]:
import io
import numpy as np
import requests
from PIL import Image
from IPython.display import display

IMG_URL = "https://labs-assets.bria.ai/sandbox-example-inputs/eraser_image_example.jpg"
MASK_URL = "https://labs-assets.bria.ai/sandbox-example-inputs/eraser_mask_example.jpg"

image = Image.open(io.BytesIO(requests.get(IMG_URL, timeout=60).content)).convert("RGB")
mask_raw = Image.open(io.BytesIO(requests.get(MASK_URL, timeout=60).content)).convert("L")
if mask_raw.size != image.size:
    mask_raw = mask_raw.resize(image.size, Image.NEAREST)
mask = Image.fromarray(((np.array(mask_raw) > 127) * 255).astype(np.uint8))
print("image", image.size, "| mask", mask.size)
display(image, mask)

## 4. Build configuration and start the pipeline

`EraseConfig()` with no arguments pulls the weights from the public `briaai/Expansion` Hugging Face repo on first `setup()` and caches them under `~/.cache/bria/erase`. (Set the `ERASE_*` environment variables from the README to run fully offline.)

In [ ]:
import torch
from erase import Erase, EraseConfig, EraseInput

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Erase.")

pipeline = Erase(config=EraseConfig())
pipeline.setup()
print("Pipeline setup complete.")

## 5. Erase the masked object

In [ ]:
import time
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

t0 = time.time()
result = pipeline.execute(EraseInput(image=image, mask=mask)).image
print(f"erased {result.size} in {time.time()-t0:.1f}s")
result.save("outputs/erased.png")
display(result)

## 6. Side-by-side comparison

input | masked region (highlighted) | erased result.

In [ ]:
inp = np.asarray(image).astype(int)
m = np.asarray(mask) > 0
overlay = inp.copy()
overlay[m] = (inp[m] * 0.35 + np.array([255, 40, 40]) * 0.65).astype(int)

h, w = inp.shape[:2]
combo = Image.new("RGB", (w * 3 + 20, h), "white")
for i, a in enumerate([image, Image.fromarray(overlay.astype(np.uint8)), result.convert("RGB")]):
    combo.paste(a, (i * (w + 10), 0))
combo.save("outputs/compare.png")
display(combo)

## 7. Cleanup

In [ ]:
pipeline.cleanup()
print("Cleanup done.")